# Tutorial 4: LoRA Finetuning on HPC4

**IEDA 4000I - Large Language Models and Operations Research**

This notebook is the class guide for Lab 4.

Important execution rule:
- Use this notebook for instructions and light checks.
- Run all GPU-critical steps in terminal (interactive SLURM session or sbatch).
- Do not rely on VSCode notebook kernel to inherit SLURM GPU allocation.

## 90-Minute Flow

| Time | Activity |
|---|---|
| 00:00-00:08 | Goals, deliverables, and grading evidence |
| 00:08-00:20 | Section 1 preflight checks + student-isolated path setup |
| 00:20-00:30 | Section 2 baseline inference on hold-out prompts |
| 00:30-00:55 | Section 3 LoRA/QLoRA training job submission (sbatch) |
| 00:55-01:10 | Section 4 load adapter and run post-finetune inference |
| 01:10-01:22 | Section 5 before vs after evaluation and error analysis |
| 01:22-01:30 | Section 6 exit ticket submission |

# Section 0: What You Will Finish Today

By the end of this lab, you should complete these core items:

1. Run one baseline inference on fixed hold-out prompts using a shared base model.
2. Submit one LoRA finetuning job with student-isolated output paths.
3. Load your trained adapter and run post-finetune inference on the same prompts.
4. Produce one before-vs-after comparison table and short conclusion.

Core deliverables:
- One training job id and key log lines
- One adapter folder path
- One before/after result table

# Section 1: Preflight and Student-Isolated Runtime Paths

Shared base models can be used by everyone for LoRA because LoRA training writes adapter weights to your own output directory.

Policy for this lab:
- Read base model from shared path (read-only).
- Never write checkpoints/logs under `/project/ugiedahpc4/ieda4000i/models/`.
- Every student uses unique output/cache/log paths.

## 1.1 Environment checks (login node)

```bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i
python --version
python -c "import torch; print('torch:', torch.__version__)"
python -c "import transformers; print('transformers:', transformers.__version__)"
python -c "import peft; print('peft:', peft.__version__)"
python -c "import datasets; print('datasets:', datasets.__version__)"
python -c "import accelerate; print('accelerate:', accelerate.__version__)"
python -c "import bitsandbytes as bnb; print('bitsandbytes:', bnb.__version__)"
```

If any package is missing, install in `ieda4000i` environment and re-check.

In [ ]:
# CPU-safe package check cell
import importlib

packages = ["torch", "transformers", "peft", "datasets", "accelerate", "bitsandbytes"]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"OK  {pkg}: {getattr(mod, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"MISSING/ERROR  {pkg}: {exc}")

## 1.2 Shared base model check

```bash
ls /project/ugiedahpc4/ieda4000i/models/
```

Recommended base model for class-scale stability:
- `/project/ugiedahpc4/ieda4000i/models/Qwen3-1.7B`

Optional TA demo model:
- `/project/ugiedahpc4/ieda4000i/models/Qwen3-4B`

## 1.3 Student-isolated path template (copy to terminal)

```bash
# Navigate to the Lab 4 directory inside your course repo.
# Replace <your-repo> with the folder name you used when cloning (e.g. IEDA-4000I-SP26).
cd ~/<your-repo>/Labs/Lab\ 4/

export COURSE_ENV=ieda4000i
export BASE_MODEL="/project/ugiedahpc4/ieda4000i/models/Qwen3-1.7B"

# unique run id per student + per run
export RUN_TAG="${USER}_$(date +%Y%m%d_%H%M%S)"

# isolate all writable paths
export LAB4_RUN_ROOT="$HOME/lab4_runs/${RUN_TAG}"
export OUTPUT_DIR="${LAB4_RUN_ROOT}/adapter"
export LOG_DIR="${LAB4_RUN_ROOT}/logs"
export HF_HOME="${LAB4_RUN_ROOT}/hf_cache"

mkdir -p "${OUTPUT_DIR}" "${LOG_DIR}" "${HF_HOME}"
echo "RUN_TAG=${RUN_TAG}"
echo "OUTPUT_DIR=${OUTPUT_DIR}"
```

This template avoids collisions even when all students train from the same base model path.

# Section 2: Baseline Inference (Before Finetune)

First, request an interactive GPU session (skip if you already have one):

```bash
srun --partition=gpu-a30 --gres=gpu:1 --account=ugiedahpc4 --time=00:30:00 --pty bash
```

Once on the GPU node, re-activate the environment and re-export your variables from Section 1.3, then run:

```bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i

# Re-export the same variables from Section 1.3 here (cd, BASE_MODEL, RUN_TAG, etc.)

# Preview prompt file
cat data/eval_prompts.jsonl | head -n 3

python scripts/run_eval.py \
  --model_path "${BASE_MODEL}" \
  --prompt_file data/eval_prompts.jsonl \
  --output_file "${LAB4_RUN_ROOT}/before.jsonl" \
  --max_new_tokens 128 \
  --temperature 0.2 \
  --device cuda
```

# Section 3: LoRA/QLoRA Finetuning via sbatch

## 3.1 Submit job

> **Note:** The sbatch script runs **standard LoRA in bf16** by default (5 epochs).
> Qwen3-1.7B fits comfortably on a single L20 GPU in full precision.
> To enable 4-bit QLoRA quantization, set `export USE_4BIT=1` before submitting.

Exit the interactive GPU session first (type `exit`), then submit from the login node:

```bash
export TRAIN_FILE=data/train_lora.jsonl
export VAL_FILE=data/val_lora.jsonl

sbatch scripts/train_lora.sbatch
```

The sbatch script reads environment variables from Section 1.3 and writes only to your isolated folders.

## 3.2 Understanding `train_lora.sbatch`

Read the actual script at `scripts/train_lora.sbatch` to understand the full configuration.

Key design features:
- **Student-isolated paths**: uses `RUN_TAG`, `OUTPUT_DIR`, `LOG_DIR`, `HF_HOME` from your environment (with safe defaults).
- **Full precision by default**: loads the base model in bf16. Set `USE_4BIT=1` for QLoRA if needed.\n
- **bf16 compute**: uses bfloat16 for stable mixed-precision training on L20 GPUs.
- **All parameters configurable**: you can override any setting via environment variables before `sbatch`.

Configurable environment variables (all have defaults):

| Variable | Default | Description |
|---|---|---|
| `BASE_MODEL` | Qwen3-1.7B shared path | Base model to fine-tune |
| `USE_4BIT` | `0` | Set `1` for QLoRA (4-bit quantization) |
| `LORA_R` | `16` | LoRA rank |
| `LORA_ALPHA` | `32` | LoRA alpha scaling |
| `NUM_TRAIN_EPOCHS` | `5` | Training epochs |
| `PER_DEVICE_TRAIN_BATCH_SIZE` | `2` | Batch size per GPU |
| `GRADIENT_ACCUMULATION_STEPS` | `8` | Gradient accumulation |
| `LEARNING_RATE` | `2e-4` | Learning rate |

## 3.2.1 Understanding `DataCollatorForLanguageModeling`

In `train_lora.py`, we tokenize each sample **without padding** (`padding=False`) to save memory — each sample keeps only its real tokens. The data collator's job is to pad samples to equal length when forming a batch.

We use `DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)` with two design choices:

1. **`mlm=False`** — because Qwen3 is a decoder-only (causal LM) model, not a masked language model.
2. **No explicit `labels` in `tokenize_fn`** — the collator creates labels automatically.

**What the collator does internally (CLM mode):**

```
Input (variable-length, no labels):
  Sample A: input_ids = [101, 202, 303]
  Sample B: input_ids = [401, 502, 603, 704]

Step 1: tokenizer.pad() → pad input_ids to max length in batch
  input_ids = [[101, 202, 303, PAD],  [401, 502, 603, 704]]

Step 2: labels = input_ids.clone()
  labels    = [[101, 202, 303, PAD],  [401, 502, 603, 704]]

Step 3: mask padding in labels → labels[labels == pad_token_id] = -100
  labels    = [[101, 202, 303, -100], [401, 502, 603, 704]]
  (cross-entropy loss ignores -100 positions → padding doesn't affect training)
```

> **Key point:** Do NOT set `labels` in your `tokenize_fn` when using this collator.
> If you add a `labels` field, `tokenizer.pad()` will try to process it as an unknown
> field and fail on variable-length sequences. Let the collator handle labels creation.

> **Why `padding=False` in tokenization?** This is called the "dynamic padding" strategy.
> Instead of padding every sample to `max_length` (wasteful), we let the collator pad each
> batch to only the longest sample in that batch, saving memory.

## 3.3 Monitor and verify

```bash
squeue -u $USER
# after completion, replace JOB_ID
cat lab4_lora_JOB_ID.out
cat lab4_lora_JOB_ID.err
ls "${OUTPUT_DIR}"
```

Expected evidence:
- training arguments printed
- loss logs printed
- adapter files exist in `${OUTPUT_DIR}`

# Section 4: Post-Finetune Inference (After Finetune)

Request a GPU interactive session if you don't have one:

```bash
srun --partition=gpu-a30 --gres=gpu:1 --account=ugiedahpc4 --time=00:15:00 --pty bash
source "/opt/shared/.spack-edge/dist/bin/setup-env.sh" -y
conda activate ieda4000i
# Re-export variables from Section 1.3
```

Run on the same prompt file to ensure fair comparison:

```bash
python scripts/run_eval.py \
  --model_path "${BASE_MODEL}" \
  --adapter_path "${OUTPUT_DIR}" \
  --prompt_file data/eval_prompts.jsonl \
  --output_file "${LAB4_RUN_ROOT}/after.jsonl" \
  --max_new_tokens 128 \
  --temperature 0.2 \
  --device cuda
```

# Section 5: Before vs After Evaluation

> **Note on dataset size:** The provided training set has 50 examples — enough to observe some
> quality differences but still small compared to production finetuning. With the default 1 epoch,
> improvements may be subtle. Try increasing `NUM_TRAIN_EPOCHS` to 3–5 for more visible changes.

Use both automatic and manual checks:
1. Automatic: format compliance or keyword hit rate.
2. Manual: correctness, completeness, instruction following, conciseness (1-5 rubric).

Passing suggestion:
- The pipeline ran successfully end-to-end, and
- At least 2 prompts show some detectable difference in output style or content.

In [ ]:
# Before vs After comparison helper (CPU-safe)
import json
from pathlib import Path

# ----- Set your run folder here -----
# Example: run_root = Path.home() / "lab4_runs" / "myitsc_20260307_140000"
run_root = Path.home() / "lab4_runs"

# Auto-detect: use the most recent run folder
run_dirs = sorted(run_root.iterdir()) if run_root.exists() else []
if run_dirs:
    latest = run_dirs[-1]
    print(f"Using latest run: {latest.name}")
else:
    latest = run_root
    print("No run folders found. Set run_root manually.")

before_file = latest / "before.jsonl"
after_file = latest / "after.jsonl"

def load_jsonl(path):
    if not path.exists():
        print(f"  [Not found] {path}")
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

before = load_jsonl(before_file)
after = load_jsonl(after_file)

if before and after:
    print(f"\n{'='*80}")
    print(f"{'ID':<6} {'Prompt':<50} {'Before (truncated)':<40} {'After (truncated)':<40}")
    print(f"{'='*80}")
    for b, a in zip(before, after):
        pid = b.get("id", "?")
        prompt = b["prompt"][:48]
        pred_b = b["prediction"][:38].replace("\n", " ")
        pred_a = a["prediction"][:38].replace("\n", " ")
        print(f"{pid:<6} {prompt:<50} {pred_b:<40} {pred_a:<40}")
    print(f"{'='*80}")
else:
    print("Run baseline and post-finetune inference first (Sections 2 and 4).")

# Section 6: Bridge to Project Work

After this lab, you should be able to:
- swap dataset and keep the same training pipeline,
- tune LoRA hyperparameters (`r`, `alpha`, `dropout`, epochs),
- track run metadata for reproducibility.

Recommended next experiments:
- Compare Qwen3-1.7B vs Qwen3-4B (same dataset and eval prompts).
- Compare LoRA vs QLoRA memory usage and throughput.

# Section 7: Exit Ticket

Submit the following evidence:
1. `BASE_MODEL` path used and why.
2. `sbatch` job id for finetuning.
3. Key lines from `lab4_lora_<jobid>.out` (training started and finished).
4. Adapter output folder path (`OUTPUT_DIR`).
5. One before/after comparison table on fixed prompts.
6. One short paragraph: what improved, what still fails.

If anything fails, include exact error lines and your attempted fix command.

## Troubleshooting Quick Notes

| Symptom | Likely cause | Quick fix |
|---|---|---|
| CUDA not available | Running on login node | Request GPU with `srun` or use `sbatch` |
| Output folder conflict | Shared/static output path | Use unique `RUN_TAG` and per-user directories |
| OOM during training | Model too large or batch too large | Use 1.7B, reduce batch size, increase grad accumulation |
| Adapter not found at inference | Wrong `OUTPUT_DIR` | `ls ${OUTPUT_DIR}` and re-run with exact path |
| Job pending too long | Partition busy | check `sinfo`, switch allowed partition |
| `ModuleNotFoundError` | Missing package in env | activate `ieda4000i`, install package, retry |